# Week 6-2 · TIO-01 — System Architecture for Automated Trading Systems
**Module:** Trading Tech, Infra & Operations · **Type:** conceptual lecture (no shipped dataset)

This lecture is about *architecture*, not a dataset — so this notebook **reproduces every quantitative
idea in the lecture as runnable Python**: the latency hierarchy (human → software → FPGA), how a
**market-data adapter** deciphers a binary exchange packet, how a **data normalizer** unifies two
exchange formats, how the **FIX protocol** builds and parses an order, the **fat-finger RMS** range
check, the **scale of market data**, and the **dispersion / implied-correlation** maths the instructor
used to motivate the event store.

Everything below runs with only the Python standard library + numpy (no external data, no yfinance).

## 1. Components of a trading system
The lecture lists what *any* trading system needs. We model them as a simple pipeline.

In [1]:
components = [
    "Market Data Adapter  (read & decipher exchange data)",
    "Data Warehouse       (storehouse of historical data)",
    "Operational Data Store(subset kept locally for this strategy)",
    "Trader's App / CEP    (where trading decisions are made)",
    "Order Manager (OMS)   (route orders + risk + 2-way comms)",
    "Risk Management (RMS)  (fat-finger + main order checks)",
    "Back-office record     (compliance: store 7 years)",
]
for i, c in enumerate(components, 1):
    print(f"{i}. {c}")

1. Market Data Adapter  (read & decipher exchange data)
2. Data Warehouse       (storehouse of historical data)
3. Operational Data Store(subset kept locally for this strategy)
4. Trader's App / CEP    (where trading decisions are made)
5. Order Manager (OMS)   (route orders + risk + 2-way comms)
6. Risk Management (RMS)  (fat-finger + main order checks)
7. Back-office record     (compliance: store 7 years)


## 2. Traditional vs automated: the response-time gap
The instructor's central motivation for going algorithmic. A human's fastest reaction to a stimulus is
~0.15 s (the iPad colour-change game). Software systems react in single-digit microseconds; pure-hardware
FPGA systems in tens of nanoseconds (~35 ns for the FPGA stage alone).

In [2]:
latencies = {
    "Human (alert)":        0.15,          # 150 ms  fastest measured tap
    "Human (typical)":      0.25,          # 250 ms
    "Software-only system": 5e-6,          # ~5 microseconds
    "FPGA hardware stage":  35e-9,          # 35 nanoseconds
}
print(f"{'system':<22}{'seconds':>14}{'vs human(150ms)':>18}")
human = latencies['Human (alert)']
for k, v in latencies.items():
    print(f"{k:<22}{v:>14.2e}{human/v:>17,.0f}x")
speedup = latencies['Human (alert)'] / latencies['FPGA hardware stage']
print(f"\nFPGA is ~{speedup:,.0f}x faster than the fastest human reaction.")

system                       seconds   vs human(150ms)
Human (alert)               1.50e-01                1x
Human (typical)             2.50e-01                1x
Software-only system        5.00e-06           30,000x
FPGA hardware stage         3.50e-08        4,285,714x

FPGA is ~4,285,714x faster than the fastest human reaction.


## 3. Hardware vs software path
Data arrives at the **LAN card** (physical layer). Either the logic runs *on the card* in an FPGA (fast,
rigid), or it travels via **PCI → CPU** where software (C++/Python) runs (flexible, slower). We add up a
toy latency budget for each path.

In [3]:
# toy per-stage latencies (nanoseconds)
software_path = {"NIC receive":80, "PCI to CPU":250, "OS/kernel stack":900,
                 "app decode":600, "strategy logic":400, "build packet":300, "PCI+NIC send":330}
fpga_path     = {"NIC receive":40, "decode on FPGA":15, "logic on FPGA":20, "build+send on FPGA":35}
sw = sum(software_path.values()); fp = sum(fpga_path.values())
print(f"Software-path total : {sw:>6} ns  ({sw/1000:.2f} us)")
print(f"FPGA-path total     : {fp:>6} ns  ({fp/1000:.3f} us)")
print(f"FPGA is {sw/fp:.1f}x faster end-to-end in this toy budget.")

Software-path total :   2860 ns  (2.86 us)
FPGA-path total     :    110 ns  (0.110 us)
FPGA is 26.0x faster end-to-end in this toy budget.


## 4. The market-data adapter: deciphering a binary packet
Each exchange sends market data in its **own fixed-byte format**. The adapter knows the layout. We build
a packet with `struct` (instrument id, bid, ask, bid-qty, ask-qty) and decode it back.

In [4]:
import struct
# layout: >I = uint32 instrument id, then 4 floats (bid, ask, bidqty, askqty)
fmt = ">Iffff"
raw = struct.pack(fmt, 14, 270.10, 270.14, 1200.0, 900.0)   # instrument 14 = 'AAPL'
print("raw bytes:", raw.hex())
instr, bid, ask, bidq, askq = struct.unpack(fmt, raw)
names = {14: "AAPL", 7: "MSFT", 3: "SBIN"}
print(f"decoded -> {names.get(instr,'?')}  bid={bid:.2f}  ask={ask:.2f}  "
      f"bidQty={bidq:.0f}  askQty={askq:.0f}  spread={ask-bid:.2f}")

raw bytes: 0000000e43870ccd438711ec4496000044610000
decoded -> AAPL  bid=270.10  ask=270.14  bidQty=1200  askQty=900  spread=0.04


## 5. The data normalizer: two exchanges, one order book
Exchange A sends best-level fields; Exchange B sends *cumulative* depth that must be de-cumulated.
The normalizer converts both into the same standard structure.

In [5]:
# Exchange A: already best bid/ask
ex_a = {"instr":"SBIN", "best_bid":542.10, "best_ask":542.35, "bidq":800, "askq":650}
# Exchange B: cumulative ask quantities at levels 1 and 1+2 -> must de-cumulate
ex_b_cum = {"instr":"SBIN", "ask_px":[542.40, 542.55], "ask_cumqty":[700, 1900]}

def normalize_a(d):
    return {"instr":d["instr"], "bid":d["best_bid"], "ask":d["best_ask"],
            "bidq":d["bidq"], "askq":d["askq"]}

def normalize_b(d):
    cum = d["ask_cumqty"]; lvl_qty = [cum[0]] + [cum[i]-cum[i-1] for i in range(1,len(cum))]
    return {"instr":d["instr"], "ask_px":d["ask_px"], "ask_levelqty":lvl_qty}

print("A normalized:", normalize_a(ex_a))
nb_ = normalize_b(ex_b_cum)
print("B de-cumulated level qty:", nb_["ask_levelqty"], "(was cumulative", ex_b_cum["ask_cumqty"], ")")

A normalized: {'instr': 'SBIN', 'bid': 542.1, 'ask': 542.35, 'bidq': 800, 'askq': 650}
B de-cumulated level qty: [700, 1200] (was cumulative [700, 1900] )


## 6. The FIX protocol: build & parse an order
FIX is a standardized **tag=value** text protocol (SOH-delimited). The lecture sent a defensive AAPL buy.
We build it and parse it back, then note *why* it is slow (text, bloaty).

In [6]:
SOH = chr(1)   # field separator
TAGS = {1:"Account",11:"ClOrdID",35:"MsgType",38:"OrderQty",44:"Price",
        48:"SecurityID",54:"Side",55:"Symbol"}
order = {1:"Q001", 11:"ORD123", 48:"AAPL", 44:265.01, 54:1, 38:100, 35:"D"}  # 54=1 buy, 35=D new
msg = SOH.join(f"{t}={v}" for t, v in order.items())
print("FIX message (| shown for SOH):")
print(msg.replace(SOH, "|"))

parsed = {int(t): v for t, v in (f.split("=") for f in msg.split(SOH))}
print("\nParsed:")
for t, v in parsed.items():
    print(f"  {t:>3} {TAGS.get(t,'?'):<11} = {v}")

# why FIX is bloaty: price as text vs as a 4-byte float
price_text_bytes = len("265.01")   # 6 chars = 6 bytes
price_float_bytes = 4
print(f"\nPrice as FIX text = {price_text_bytes} bytes vs binary float = {price_float_bytes} bytes "
      f"({price_text_bytes/price_float_bytes:.1f}x bigger).")

FIX message (| shown for SOH):
1=Q001|11=ORD123|48=AAPL|44=265.01|54=1|38=100|35=D

Parsed:
    1 Account     = Q001
   11 ClOrdID     = ORD123
   48 SecurityID  = AAPL
   44 Price       = 265.01
   54 Side        = 1
   38 OrderQty    = 100
   35 MsgType     = D

Price as FIX text = 6 bytes vs binary float = 4 bytes (1.5x bigger).


## 7. Fat-finger RMS: the typo guard
The trader meant IV trigger 30.01% but fat-fingered 3.01%. The within-app RMS rejects values outside a
logical range, **before** the order ever reaches the order manager.

In [7]:
RANGES = {"iv_trigger_pct": (10.0, 80.0), "order_qty": (1, 5000), "price": (0.01, 100000)}

def rms_check(param, value):
    lo, hi = RANGES[param]
    ok = lo <= value <= hi
    return ok, f"{param}={value} {'ACCEPT' if ok else f'REJECT (outside {lo}-{hi})'}"

for v in [30.01, 3.01, 95.0]:
    print(rms_check("iv_trigger_pct", v)[1])

iv_trigger_pct=30.01 ACCEPT
iv_trigger_pct=3.01 REJECT (outside 10.0-80.0)
iv_trigger_pct=95.0 REJECT (outside 10.0-80.0)


## 8. The scale of the problem
Why an operational data store (a *subset*) is essential: each exchange lists 100,000+ instruments and
emits a few hundred GB/day, but a pairs strategy may watch only 2.

In [8]:
instruments_total = 100_000
data_per_day_gb   = 300            # "few hundred GB/day", compressed binary
watched           = 2             # e.g. crude month-1 vs month-2
tick_per_asset_mb = 200           # FAQ: a few hundred MB tick-by-tick / asset / day
print(f"Instruments listed      : {instruments_total:,}")
print(f"Market data per day     : ~{data_per_day_gb} GB (compressed)")
print(f"Instruments you watch   : {watched}  ->  {watched/instruments_total:.5%} of the universe")
print(f"Tick data, 1 asset/day  : ~{tick_per_asset_mb} MB")
print("=> keep a tiny operational-data-store subset, not the whole firehose.")

Instruments listed      : 100,000
Market data per day     : ~300 GB (compressed)
Instruments you watch   : 2  ->  0.00200% of the universe
Tick data, 1 asset/day  : ~200 MB
=> keep a tiny operational-data-store subset, not the whole firehose.


## 9. Dispersion strategy & implied correlation
The lecture's deep example. Index variance is a function of component variances, weights, and the
**pairwise correlation**. Assuming one *constant* correlation rho across all pairs:

$$\sigma_I^2 = \sum_i w_i^2 \sigma_i^2 \; + \; \rho \sum_{i \neq j} w_i w_j \sigma_i \sigma_j$$

Solve for the **implied correlation** the market is pricing, then decide the trade.

In [9]:
import numpy as np
w     = np.array([0.5, 0.5])          # two 50% components
sig_i = np.array([0.20, 0.20])        # component vols (annualised)
sig_I = 0.16                          # index vol implied from index options

# build the cross term sum_{i!=j} w_i w_j sig_i sig_j
M = np.outer(w*sig_i, w*sig_i)
diag = np.sum(np.diag(M))             # = sum w_i^2 sig_i^2
cross = np.sum(M) - diag              # sum over i!=j
implied_rho = (sig_I**2 - diag) / cross
print(f"diagonal (sum w^2 sig^2) = {diag:.4f}")
print(f"cross term coefficient   = {cross:.4f}")
print(f"Implied correlation rho  = {implied_rho:.3f}")

# decision rule from the lecture
if implied_rho < 0.5:
    trade = "market prices LOW correlation -> if it rises, index vol rises: BUY index vol, SELL stock vol"
else:
    trade = "market prices HIGH correlation -> if it falls, index vol falls: SELL index vol, BUY stock vol"
print("Trade:", trade)

diagonal (sum w^2 sig^2) = 0.0200


cross term coefficient   = 0.0200
Implied correlation rho  = 0.280
Trade: market prices LOW correlation -> if it rises, index vol rises: BUY index vol, SELL stock vol


### 9b. Sanity check: rho = +1 and rho = -1 bracket the index vol
With two equal 50/50 components at 20% vol, perfectly correlated index vol = 20%, perfectly
anti-correlated index vol = 0% (the flat index). Our 16% sits in between.

In [10]:
def index_vol(rho):
    var = diag + rho*cross
    return np.sqrt(max(var, 0.0))
for r in [1.0, 0.5, implied_rho, 0.0, -1.0]:
    print(f"rho={r:+.2f} -> index vol = {index_vol(r):.3f}")

rho=+1.00 -> index vol = 0.200
rho=+0.50 -> index vol = 0.173
rho=+0.28 -> index vol = 0.160
rho=+0.00 -> index vol = 0.141
rho=-1.00 -> index vol = 0.000


## 10. Quantified news: the four scores
Vendors (Thomson Reuters, RavenPack/Dow Jones, Bloomberg) attach scores to each article so machines can
act. We model the four the instructor named and a tiny novelty de-duplicator.

In [11]:
articles = [
    {"id":1, "text":"CEO ABC fined $20 million by regulator", "companies":["ABC"]},
    {"id":2, "text":"CEO ABC fined $20 million, to pay in two months", "companies":["ABC"]},
    {"id":3, "text":"XYZ posts record quarterly profit, shares jump", "companies":["XYZ"]},
]
pos_words = {"profit","record","jump","beat","up","gain"}
neg_words = {"fined","loss","fraud","down","cut"}

def sentiment(t):
    toks = set(t.lower().replace(",", " ").replace("$"," ").split())
    p, n = len(toks & pos_words), len(toks & neg_words)
    return (p - n) / max(p + n, 1)

seen = []   # novelty: reject if too similar to a prior article on same company
def novelty(t, companies):
    for s in seen:
        if companies == s[0] and len(set(t.lower().split()) & set(s[1].lower().split())) >= 4:
            return 0.0
    seen.append((companies, t)); return 1.0

for a in articles:
    s = sentiment(a["text"]); nov = novelty(a["text"], a["companies"])
    print(f"id{a['id']}  sentiment={s:+.2f}  relevance=1.0(single co)  novelty={nov:.0f}  "
          f"{'ACT' if nov>0 else 'SKIP (repeat)'}  :: {a['text']}")

id1  sentiment=-1.00  relevance=1.0(single co)  novelty=1  ACT  :: CEO ABC fined $20 million by regulator
id2  sentiment=-1.00  relevance=1.0(single co)  novelty=0  SKIP (repeat)  :: CEO ABC fined $20 million, to pay in two months
id3  sentiment=+1.00  relevance=1.0(single co)  novelty=1  ACT  :: XYZ posts record quarterly profit, shares jump


## 11. The simulator & data replay
A new strategy is *not* sent live. It is run against an **exchange simulator** fed by **replayed historical
data** (flat day, volatile day, gap-up...). We model a probabilistic fill for a passive order sitting
between bid and ask.

In [12]:
import numpy as np
rng = np.random.default_rng(7)

def fill_probability(order_px, bid, ask):
    # aggressive (>= ask) -> ~certain; passive deep below bid -> ~0; in-spread -> linear-ish
    if order_px >= ask: return 0.99
    if order_px <= bid: return 0.05
    frac = (order_px - bid) / (ask - bid)     # 0 at bid, 1 at ask
    return 0.05 + 0.9*frac

bid, ask = 99.00, 100.00
for px in [98.50, 99.00, 99.50, 99.99, 100.00]:
    p = fill_probability(px, bid, ask)
    fills = rng.random(10000) < p
    print(f"order@{px:>6.2f}  P(fill)={p:.2f}  simulated fill rate={fills.mean():.2f}")

order@ 98.50  P(fill)=0.05  simulated fill rate=0.05
order@ 99.00  P(fill)=0.05  simulated fill rate=0.05
order@ 99.50  P(fill)=0.50  simulated fill rate=0.50
order@ 99.99  P(fill)=0.94  simulated fill rate=0.94
order@100.00  P(fill)=0.99  simulated fill rate=0.99


## Key takeaways
- A trading system = **market-data adapter → (data warehouse / operational data store) → trader's app/CEP → order manager (with RMS) → exchange**, plus back-office record-keeping (7 yrs).
- Going algorithmic closes a **~4-million-to-one** reaction-time gap (human 150 ms vs FPGA 35 ns) and scales from ~50 to tens of thousands of instruments.
- **Hardware (FPGA on the NIC)** is fast but rigid; **software (PCI→CPU)** is flexible but slower.
- Each exchange speaks its **own binary language**; the **adapter** deciphers it and the **normalizer** unifies formats; **FIX** is a standard but text-based (bloaty, slower).
- **RMS** lives both in the app (fat-finger typo guard) and — more importantly — once, just before orders leave the **order manager**.
- The **dispersion** example: index variance = component variances + pairwise correlation; solving for the constant **implied correlation** tells you whether to buy or sell index vol vs stock vol.
- Strategies are validated on an **exchange simulator** with **replayed data** before going live; good simulators are proprietary and demand deep market-microstructure knowledge.